# \-----------------------------------Workshop 5 -------------------------------------- (222 Words)

## T5.1: Download the 'Portugal_online_retail', 'Sweden_online_retail, and 'UK_online_retail' datasets from Canvas. Apply the apriori algorithm to all datasets using three different confidence levels. Select one confidence level for each dataset that you think works better. Determine the first three most important rules for each dataset using the selected confidence level and report them in the report cell. Explain what each rule means (Completing the report cell is required) (15%).

NOTE: You should comment on your code

In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.preprocessing import LabelEncoder
import warnings

# Suppressing warnings 
warnings.filterwarnings("ignore")

# Defining dataset paths
dataset_paths = {
    'Portugal': "https://raw.githubusercontent.com/maz786/WlvMScAI/main/7CS033DataMining/Week5/Workshop/Portugal_online_retail.csv",
    'Sweden': "https://raw.githubusercontent.com/maz786/WlvMScAI/main/7CS033DataMining/Week5/Workshop/Sweden_online_retail.csv",
    'UK': "https://media.githubusercontent.com/media/maz786/WlvMScAI/main/7CS033DataMining/Week5/Workshop/UK_online_retail.csv"
}

def preprocess_dataset(path):
    """Load and preprocess dataset."""
    df = pd.read_csv(path, dtype='unicode')
    df.drop('InvoiceNo', axis=1, inplace=True, errors='ignore')
    
    # Encoding data categorically
    categorical_cols = df.select_dtypes(include=['object']).columns
    if len(categorical_cols) > 0:
        encoder = LabelEncoder()
        for col in categorical_cols:
            df[col] = encoder.fit_transform(df[col].astype(str))
            
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
    return df

def find_best_rules(df, dataset_name):
    """Apply Apriori algorithm with different confidence levels and select the best one."""
    # Min_support adjustment for each dataset accordingly
    min_support = 0.02 if dataset_name == 'UK' else 0.05
    
    best_average_lift = -1
    best_confidence = None
    best_rules = None
    
    for confidence in [0.1, 0.2, 0.3]:
        frq_items = apriori(df, min_support=min_support, use_colnames=True)
        rules = association_rules(frq_items, metric="confidence", min_threshold=confidence)
        rules['length'] = rules['antecedents'].apply(lambda x: len(x))
        
        # Filtering rules with more than one antecedent for more interesting insights
        rules = rules[rules['length'] > 1]
        
        if not rules.empty and rules['lift'].mean() > best_average_lift:
            best_average_lift = rules['lift'].mean()
            best_confidence = confidence
            best_rules = rules
    
    print(f"\nBest confidence for {dataset_name}: {best_confidence}")
    if best_rules is not None:
        best_rules.sort_values('lift', ascending=False, inplace=True)
        return best_rules.head(3)
    else:
        return pd.DataFrame()

# Dataset analysation
for dataset_name, path in dataset_paths.items():
    print(f"\nAnalyzing {dataset_name} dataset...")
    df = preprocess_dataset(path)
    top_rules = find_best_rules(df, dataset_name)
    if not top_rules.empty:
        print(f"Top 3 rules for {dataset_name}:\n{top_rules}")
    else:
        print(f"No significant rules found for {dataset_name}.")


Analyzing Portugal dataset...



Best confidence for Portugal: 0.1
Top 3 rules for Portugal:
                                              antecedents  \
163949  (CHARLOTTE BAG SUKI DESIGN, JUMBO BAG RED RETR...   
306121  (JUMBO BAG RED RETROSPOT, JUMBO SHOPPER VINTAG...   
138809  (PLASTERS IN TIN VINTAGE PAISLEY, JUMBO BAG SC...   

                                              consequents  antecedent support  \
163949  (LUNCH BAG PINK POLKADOT, JUMBO  BAG BAROQUE B...            0.051724   
306121  (LUNCH BAG VINTAGE LEAF DESIGN, JUMBO BAG SCAN...            0.051724   
138809  (PACK OF 12 RED RETROSPOT TISSUES, SET/3 RED G...            0.051724   

        consequent support   support  confidence       lift  leverage  \
163949            0.051724  0.051724         1.0  19.333333  0.049049   
306121            0.051724  0.051724         1.0  19.333333  0.049049   
138809            0.051724  0.051724         1.0  19.333333  0.049049   

        conviction  zhangs_metric  length  
163949         inf            1.


Best confidence for Sweden: 0.1
Top 3 rules for Sweden:
                                             antecedents  \
615    (36 DOILIES DOLLY GIRL, 60 CAKE CASES DOLLY GI...   
70956  (BAG 250g SWIRLY MARBLES, SET OF 3 CAKE TINS P...   
70954  (RETROSPOT TEA SET CERAMIC 11 PC, SET OF 3 CAK...   

                                             consequents  antecedent support  \
615                       (ASSORTED BOTTLE TOP  MAGNETS)            0.055556   
70956  (VINTAGE HEADS AND TAILS CARD GAME, RETROSPOT ...            0.055556   
70954                          (BAG 250g SWIRLY MARBLES)            0.055556   

       consequent support   support  confidence  lift  leverage  conviction  \
615              0.055556  0.055556         1.0  18.0  0.052469         inf   
70956            0.055556  0.055556         1.0  18.0  0.052469         inf   
70954            0.055556  0.055556         1.0  18.0  0.052469         inf   

       zhangs_metric  length  
615              1.0       2  
70


Best confidence for UK: 0.1
Top 3 rules for UK:
                                           antecedents  \
166  (GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...   
165  (PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY...   
164  (PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY...   

                           consequents  antecedent support  \
166   (PINK REGENCY TEACUP AND SAUCER)            0.037553   
165  (GREEN REGENCY TEACUP AND SAUCER)            0.029249   
164  (ROSES REGENCY TEACUP AND SAUCER)            0.030910   

     consequent support  support  confidence       lift  leverage  conviction  \
166            0.037660  0.02641    0.703281  18.674462  0.024996    3.243271   
165            0.050035  0.02641    0.902930  18.046041  0.024947    9.786434   
164            0.051267  0.02641    0.854419  16.666089  0.024826    6.516893   

     zhangs_metric  length  
166       0.983380       2  
165       0.973047       2  
164       0.969980       2  


1. <span style="background-color: rgb(255, 255, 255); color: rgb(13, 13, 13); font-family: Söhne, ui-sans-serif, system-ui, -apple-system, &quot;Segoe UI&quot;, Roboto, Ubuntu, Cantarell, &quot;Noto Sans&quot;, sans-serif, &quot;Helvetica Neue&quot;, Arial, &quot;Apple Color Emoji&quot;, &quot;Segoe UI Emoji&quot;, &quot;Segoe UI Symbol&quot;, &quot;Noto Color Emoji&quot;; font-size: 16px; white-space-collapse: preserve;">Suppresses Warnings: Code suppresses warnings for clean output.</span>
2. <span style="background-color: rgb(255, 255, 255); color: rgb(13, 13, 13); font-family: Söhne, ui-sans-serif, system-ui, -apple-system, &quot;Segoe UI&quot;, Roboto, Ubuntu, Cantarell, &quot;Noto Sans&quot;, sans-serif, &quot;Helvetica Neue&quot;, Arial, &quot;Apple Color Emoji&quot;, &quot;Segoe UI Emoji&quot;, &quot;Segoe UI Symbol&quot;, &quot;Noto Color Emoji&quot;; font-size: 16px; white-space-collapse: preserve;">Defines Dataset Paths: Specifies paths for Portugal, Sweden, and UK datasets.</span>
3. <span style="background-color: rgb(255, 255, 255); color: rgb(13, 13, 13); font-family: Söhne, ui-sans-serif, system-ui, -apple-system, &quot;Segoe UI&quot;, Roboto, Ubuntu, Cantarell, &quot;Noto Sans&quot;, sans-serif, &quot;Helvetica Neue&quot;, Arial, &quot;Apple Color Emoji&quot;, &quot;Segoe UI Emoji&quot;, &quot;Segoe UI Symbol&quot;, &quot;Noto Color Emoji&quot;; font-size: 16px; white-space-collapse: preserve;">Preprocesses Datasets: preprocess_dataset function loads data, removes 'InvoiceNo', encodes categorical data, converts to numeric, and replaces NaNs with 0.</span>
4. <span style="background-color: rgb(255, 255, 255); color: rgb(13, 13, 13); font-family: Söhne, ui-sans-serif, system-ui, -apple-system, &quot;Segoe UI&quot;, Roboto, Ubuntu, Cantarell, &quot;Noto Sans&quot;, sans-serif, &quot;Helvetica Neue&quot;, Arial, &quot;Apple Color Emoji&quot;, &quot;Segoe UI Emoji&quot;, &quot;Segoe UI Symbol&quot;, &quot;Noto Color Emoji&quot;; font-size: 16px; white-space-collapse: preserve;">Finds Best Rules: Using Apriori method, find_best_rules function identifies frequent itemsets with varying support thresholds and computes association rules at different confidence levels. It refines rules based on antecedents count, selects optimal confidence level, and returns top three rules based on lift.</span>
5. <span style="background-color: rgb(255, 255, 255); color: rgb(13, 13, 13); font-family: Söhne, ui-sans-serif, system-ui, -apple-system, &quot;Segoe UI&quot;, Roboto, Ubuntu, Cantarell, &quot;Noto Sans&quot;, sans-serif, &quot;Helvetica Neue&quot;, Arial, &quot;Apple Color Emoji&quot;, &quot;Segoe UI Emoji&quot;, &quot;Segoe UI Symbol&quot;, &quot;Noto Color Emoji&quot;; font-size: 16px; white-space-collapse: preserve;">Analyzes Each Dataset: Iteratively preprocesses and analyzes each dataset, displaying top rules based on chosen confidence level. (97 words)</span>

<span style="background-color: rgb(255, 255, 255); color: rgb(13, 13, 13); font-family: Söhne, ui-sans-serif, system-ui, -apple-system, &quot;Segoe UI&quot;, Roboto, Ubuntu, Cantarell, &quot;Noto Sans&quot;, sans-serif, &quot;Helvetica Neue&quot;, Arial, &quot;Apple Color Emoji&quot;, &quot;Segoe UI Emoji&quot;, &quot;Segoe UI Symbol&quot;, &quot;Noto Color Emoji&quot;; font-size: 16px; white-space-collapse: preserve;">Association rule mining identifies recurring patterns in databases, aiding understanding of client behavior in retail. For instance, in Portuguese data, buyers purchasing "JUMBO BAG RED RETROSPOT" and "JUMBO BAG PINK VINTAGE PAISLEY" are likely to buy "JUMBO BAG SCANDINAVIAN BLUE PAISLEY," useful for product placement. Swedish data shows correlation between "36 DOILIES DOLLY GIRL" and "PACK OF 60 PINK PAISLEY CAKE CASES." UK data indicates preferences for "GREEN REGENCY TEACUP AND SAUCER" and "ROSES REGENCY TEACUP AND SAUCER," suggesting "PINK REGENCY TEACUP AND SAUCER" follow-up. Metrics like support, confidence, and lift quantify rule strength. Essential sources include Agrawal et al. (1993), Han et al. (2011), and Hastie et al. (2009), offering foundational knowledge. Practical application insights by Hahsler et al. (2005) provide real-world utilization insights.</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">&nbsp;(125 words)</span>  

Bibliography

Agrawal, R., Imieliński, T., & Swami, A. (1993). Mining association rules between sets of items in large databases. In Proceedings of the 1993 ACM SIGMOD International Conference on Management of Data (pp. 207-216).

Han, J., Pei, J., & Kamber, M. (2011). Data Mining: Concepts and Techniques. Morgan Kaufmann.

Hastie, T., Tibshirani, R., & Friedman, J. (2009). The Elements of Statistical Learning. Springer Series in Statistics.

Hahsler, M., Grün, B., & Hornik, K. (2005). arules – A Computational Environment for Mining Association Rules and Frequent Item Sets. Journal of Statistical Software, 14(15).